# 14 · Abschlussprojekt: Tic-Tac-Toe

**Lernziele**

- Du zerlegst ein Problem in kleine Funktionen.
- Du kombinierst Listen, Schleifen, Bedingungen, Funktionen und Fehlerbehandlung.
- Du machst aus dem Notebook-Code ein lauffähiges Skript.
- Du sicherst die Spiellogik mit pytest-Tests ab.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 13 und 16.

## Die Aufgabe

Zwei Spieler setzen abwechselnd `X` und `O` auf ein 3×3-Feld. Wer zuerst drei in einer Reihe, Spalte oder Diagonale hat, gewinnt. Sind alle Felder belegt, endet das Spiel unentschieden.

Die Felder sind von 1 bis 9 nummeriert:

```
 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9
```

Wir bauen das Spiel in fünf Schritten. Jeder Schritt ist eine kleine Funktion, die du einzeln ausprobieren kannst. Im sechsten Schritt sicherst du die Spiellogik mit Tests ab.

## Schritt 1: Das Spielfeld

Das Feld ist eine Liste aus drei Zeilen, jede Zeile eine Liste aus drei Zellen. Ein leeres Feld ist `" "`.

**Übung 1 (Das Spielfeld, leicht):** Schreibe `create_board()`, die ein leeres Spielfeld zurückgibt. Vorher eine Denkaufgabe: Setze in `[[EMPTY] * 3] * 3` ein X in Feld 1 und gib das Feld aus. Welche Ausgabe entsteht, und warum?

- Vorhersagen: Schreibe deine Antwort auf, bevor du die nächste Zelle ausführst.
- Ausführen: Führe die Zelle aus und vergleiche.
- Untersuchen: Was liefert `wrong[0] is wrong[1]`?
- Ändern: Erzeuge mit derselben Comprehension ein 4×4-Feld.

In [ ]:
EMPTY = " "

wrong = [[EMPTY] * 3] * 3
wrong[0][0] = "X"
print(wrong)

In [ ]:
# Vorhersage: In jeder Zeile steht ein X, weil * 3 dreimal dieselbe Zeile wiederholt.
print(wrong[0] is wrong[1])  # True: alle drei Einträge zeigen auf dieselbe Liste


def create_board() -> list[list[str]]:
    return [[EMPTY for _ in range(3)] for _ in range(3)]


board = create_board()
board[0][0] = "X"
print(board)  # nur die erste Zeile hat ein X

# Ändern: 4x4-Feld mit derselben Comprehension
print([[EMPTY for _ in range(4)] for _ in range(4)])

## Schritt 2: Das Feld anzeigen

**Übung 2 (Das Feld anzeigen, leicht):** Schreibe `print_board(board)`, die das Feld wie oben gezeigt ausgibt.

In [ ]:
def print_board(board: list[list[str]]) -> None:
    for row_index, row in enumerate(board):
        print(" " + " | ".join(row))
        if row_index < 2:
            print("---+---+---")


board = create_board()
board[1][1] = "X"
print_board(board)

## Schritt 3: Einen Zug setzen

Feld `1` bis `9` muss in Zeile und Spalte umgerechnet werden. Feld 6 liegt z. B. in Zeile 1, Spalte 2 (gezählt ab 0). Tipp: `divmod(field - 1, 3)` liefert beides.

**Übung 3 (Einen Zug setzen, mittel):** Schreibe `place_mark(board, field, mark)`. Sie setzt das Zeichen und löst einen `ValueError` aus, wenn das Feld außerhalb von 1–9 liegt oder schon belegt ist.

In [ ]:
def place_mark(board: list[list[str]], field: int, mark: str) -> None:
    if not 1 <= field <= 9:
        raise ValueError(f"Feld {field} gibt es nicht")
    row, col = divmod(field - 1, 3)
    if board[row][col] != EMPTY:
        raise ValueError(f"Feld {field} ist schon belegt")
    board[row][col] = mark


board = create_board()
place_mark(board, 1, "X")
place_mark(board, 5, "O")
print_board(board)
try:
    place_mark(board, 5, "X")
except ValueError as error:
    print(error)

## Schritt 4: Gewinner prüfen

Es gibt acht Gewinnlinien: drei Zeilen, drei Spalten, zwei Diagonalen.

**Übung 4 (Gewinner prüfen, mittel):** Schreibe `has_won(board, mark)`, die `True` liefert, wenn `mark` eine komplette Linie hat. Schreibe außerdem `is_full(board)`.

In [ ]:
def has_won(board: list[list[str]], mark: str) -> bool:
    lines = []
    lines.extend(board)  # Zeilen
    lines.extend([[board[row][col] for row in range(3)] for col in range(3)])  # Spalten
    lines.append([board[i][i] for i in range(3)])  # Diagonale ↘
    lines.append([board[i][2 - i] for i in range(3)])  # Diagonale ↙
    return any(line == [mark, mark, mark] for line in lines)


def is_full(board: list[list[str]]) -> bool:
    return all(cell != EMPTY for row in board for cell in row)


board = create_board()
for field in [1, 5, 9]:
    place_mark(board, field, "X")
print_board(board)
print("X gewinnt:", has_won(board, "X"))
print("O gewinnt:", has_won(board, "O"))
print("Voll:", is_full(board))

## Schritt 5: Die Spielschleife

**Übung 5 (Die Spielschleife, schwer):** Schreibe `play()`. Abwechselnd wird nach einem Feld gefragt. Ungültige Eingaben (keine Zahl, belegtes Feld) werden gemeldet und erneut abgefragt. Nach jedem Zug wird geprüft, ob jemand gewonnen hat oder das Feld voll ist.

In [ ]:
def ask_for_field(mark: str) -> int:
    while True:
        answer = input(f"Spieler {mark}, welches Feld (1-9)? ")
        try:
            return int(answer)
        except ValueError:
            print(f"'{answer}' ist keine Zahl")


def play() -> None:
    board = create_board()
    mark = "X"
    while True:
        print_board(board)
        try:
            place_mark(board, ask_for_field(mark), mark)
        except ValueError as error:
            print(error)
            continue

        if has_won(board, mark):
            print_board(board)
            print(f"Spieler {mark} gewinnt!")
            return
        if is_full(board):
            print_board(board)
            print("Unentschieden!")
            return
        mark = "O" if mark == "X" else "X"


play()

## Vom Notebook zum Skript

Kopiere alle Funktionen in eine Zelle, ergänze `main()` und schreibe die erste Zeile `%%writefile tictactoe.py` darüber. Beim Ausführen legt Jupyter die Datei `tictactoe.py` neben diesem Notebook an:

```
%%writefile tictactoe.py
EMPTY = " "

...


def main():
    play()


if __name__ == "__main__":
    main()
```

Im Terminal (im Hauptordner des Kursprojekts):

```
uv run python notebooks/tictactoe.py
```

Eine fertige Version liegt in `projekte/tictactoe.py`.

In [ ]:
%%writefile tictactoe.py
"""Tic-Tac-Toe für zwei Spieler im Terminal."""

EMPTY = " "
Board = list[list[str]]


def create_board() -> Board:
    return [[EMPTY for _ in range(3)] for _ in range(3)]


def print_board(board: Board) -> None:
    for row_index, row in enumerate(board):
        print(" " + " | ".join(row))
        if row_index < 2:
            print("---+---+---")


def place_mark(board: Board, field: int, mark: str) -> None:
    if not 1 <= field <= 9:
        raise ValueError(f"Feld {field} gibt es nicht")
    row, col = divmod(field - 1, 3)
    if board[row][col] != EMPTY:
        raise ValueError(f"Feld {field} ist schon belegt")
    board[row][col] = mark


def has_won(board: Board, mark: str) -> bool:
    lines = []
    lines.extend(board)
    lines.extend([[board[row][col] for row in range(3)] for col in range(3)])
    lines.append([board[i][i] for i in range(3)])
    lines.append([board[i][2 - i] for i in range(3)])
    return any(line == [mark, mark, mark] for line in lines)


def is_full(board: Board) -> bool:
    return all(cell != EMPTY for row in board for cell in row)


def ask_for_field(mark: str) -> int:
    while True:
        answer = input(f"Spieler {mark}, welches Feld (1-9)? ")
        try:
            return int(answer)
        except ValueError:
            print(f"'{answer}' ist keine Zahl")


def play() -> None:
    board = create_board()
    mark = "X"
    while True:
        print_board(board)
        try:
            place_mark(board, ask_for_field(mark), mark)
        except ValueError as error:
            print(error)
            continue

        if has_won(board, mark):
            print_board(board)
            print(f"Spieler {mark} gewinnt!")
            return
        if is_full(board):
            print_board(board)
            print("Unentschieden!")
            return
        mark = "O" if mark == "X" else "X"


def main() -> None:
    play()


if __name__ == "__main__":
    main()

## Schritt 6: Die Spiellogik testen

Die Bausteine `create_board`, `place_mark`, `has_won` und `is_full` sind Funktionen ohne Ein- und Ausgabe und lassen sich direkt testen. Genau deshalb haben wir das Spiel so zerlegt. Wie du auch die Eingabe testest, zeigt Übung 9.

Die Tests importieren deine eigene `tictactoe.py` neben diesem Notebook. Fehlt sie, findet pytest über die Einstellung `pythonpath` in `pyproject.toml` die fertige Version in `projekte/tictactoe.py` und testet dann nicht deinen Code.

**Übung 6 (Die Spiellogik testen, mittel):** Schreibe `test_tictactoe.py` mit pytest-Tests (Ordner `testen/`, Teil 13) für:

- ein neues Feld ist leer und hat keinen Gewinner,
- `place_mark` trifft die richtige Zelle (Feld 6 ist Zeile 1, Spalte 2),
- ungültige und belegte Felder lösen einen `ValueError` aus,
- jede Art von Gewinnlinie gewinnt: Zeile, Spalte, beide Diagonalen,
- zwei in einer Reihe gewinnen noch nicht,
- ein volles Feld ohne Gewinner.

Benenne die Gewinnlinien mit `pytest.param(..., id="zeile")`, damit `pytest -v` sprechende Namen statt `[fields0]` zeigt.

Tipp: Eine kleine Hilfsfunktion, die ein Feld mit vorgegebenen Zügen füllt, spart viel Tipparbeit.

In [ ]:
%%writefile test_tictactoe.py
import pytest

from tictactoe import create_board, has_won, is_full, place_mark


def board_with(moves: dict[int, str]) -> list[list[str]]:
    """Hilfsfunktion: ein Feld mit vorgegebenen Zügen, z. B. {1: "X", 5: "O"}."""
    board = create_board()
    for field, mark in moves.items():
        place_mark(board, field, mark)
    return board


def test_new_board_is_empty():
    board = create_board()
    assert not is_full(board)
    assert not has_won(board, "X")


def test_place_mark_sets_correct_cell():
    board = board_with({6: "X"})
    assert board[1][2] == "X"


@pytest.mark.parametrize("field", [0, 10, -1])
def test_place_mark_rejects_invalid_field(field):
    with pytest.raises(ValueError, match="gibt es nicht"):
        place_mark(create_board(), field, "X")


def test_place_mark_rejects_occupied_field():
    board = board_with({5: "X"})
    with pytest.raises(ValueError, match="belegt"):
        place_mark(board, 5, "O")


@pytest.mark.parametrize(
    "fields",
    [
        pytest.param([1, 2, 3], id="zeile"),
        pytest.param([2, 5, 8], id="spalte"),
        pytest.param([1, 5, 9], id="diagonale-fallend"),
        pytest.param([3, 5, 7], id="diagonale-steigend"),
    ],
)
def test_three_in_a_line_wins(fields):
    board = board_with({field: "X" for field in fields})
    assert has_won(board, "X")
    assert not has_won(board, "O")


def test_two_in_a_line_do_not_win():
    assert not has_won(board_with({1: "X", 2: "X"}), "X")


def test_full_board_without_winner():
    # X O X
    # X O O
    # O X X
    moves = {1: "X", 2: "O", 3: "X", 4: "X", 5: "O", 6: "O", 7: "O", 8: "X", 9: "X"}
    board = board_with(moves)
    assert is_full(board)
    assert not has_won(board, "X")
    assert not has_won(board, "O")

In [ ]:
import sys

!{sys.executable} -m pytest -v test_tictactoe.py

## Erweiterungen für Schnelle

Die folgenden Übungen bauen dein Spiel in `tictactoe.py` aus. Lass nach jeder Änderung die Tests aus Schritt 6 laufen.

**Übung 7 (Spieler als Enum, mittel):** Ersetze die Strings `"X"` und `"O"` durch eine `Player(StrEnum)` mit einer Methode `other()`. Die Tests aus Schritt 6 müssen dabei grün bleiben.

Das lernst du: Code unter dem Schutz von Tests umbauen; `StrEnum` vergleicht sich wie Text, deshalb laufen die alten Tests weiter.

In [ ]:
from enum import StrEnum


class Player(StrEnum):
    X = "X"
    O = "O"

    def other(self) -> Player:
        return Player.O if self is Player.X else Player.X


mark = Player.X
print(mark, mark.other(), mark == "X")  # X O True

# In tictactoe.py: mark = Player.X statt "X" und mark = mark.other() statt der
# if-Zeile. Die alten Tests vergleichen mit "X" und bleiben grün:
board = create_board()
for field in [1, 2, 3]:
    place_mark(board, field, Player.X)
print(has_won(board, "X"))  # True

**Übung 8 (Spielstatistik mit Datum, schwer):** Schreibe nach jeder Partie eine Zeile in `ergebnisse.csv` (ISO-Zeitstempel, Spieler X, Spieler O, Ergebnis). Gib beim Start die Bilanz der letzten 7 Tage aus.

Tipp: Zum Zählen gibt es `Counter` aus der Standardbibliothek. Nach `from collections import Counter` zählt `Counter(["X", "O", "X"])` jedes Element, und `.most_common()` liefert die Paare aus Wert und Anzahl, häufigste zuerst.

Das lernst du: an eine CSV-Datei anhängen (Notebook 10), Zeitstempel mit `isoformat` und `fromisoformat` speichern und lesen, mit `timedelta` filtern (Notebook 11) und mit `Counter` zählen.

In [ ]:
import csv
import tempfile
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path

# Im eigenen Spiel: path = Path("ergebnisse.csv"). Hier ein leerer Ordner,
# damit das Notebook bei jedem Lauf frisch beginnt.
path = Path(tempfile.mkdtemp()) / "ergebnisse.csv"
FIELDNAMES = ["zeit", "x", "o", "ergebnis"]


def save_result(path: Path, player_x: str, player_o: str, result: str) -> None:
    is_new = not path.exists()
    with path.open("a", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=FIELDNAMES)
        if is_new:
            writer.writeheader()  # Kopfzeile nur beim ersten Mal
        writer.writerow(
            {
                "zeit": datetime.now().isoformat(timespec="seconds"),
                "x": player_x,
                "o": player_o,
                "ergebnis": result,
            }
        )


def recent_results(path: Path, days: int = 7) -> Counter:
    since = datetime.now() - timedelta(days=days)
    with path.open(encoding="utf-8", newline="") as file:
        recent = [row for row in csv.DictReader(file) if datetime.fromisoformat(row["zeit"]) >= since]
    return Counter(row["ergebnis"] for row in recent)


save_result(path, "Ada", "Tim", "X")
save_result(path, "Ada", "Tim", "unentschieden")
save_result(path, "Ada", "Tim", "X")
print(path.read_text(encoding="utf-8"))
print(recent_results(path).most_common())  # [('X', 2), ('unentschieden', 1)]

**Übung 9 (Eingabe testen, mittel):** Teste `ask_for_field` mit `monkeypatch` (Antworten `"abc"`, dann `"5"`) und prüfe mit `capsys` die Meldung "ist keine Zahl".

Das lernst du: Das Testen von Ein- und Ausgabe aus Teil 13 (`testen/`, Übung 3) am eigenen Projekt anwenden.

In [ ]:
%%writefile test_eingabe.py
from tictactoe import ask_for_field


def test_ask_for_field_repeats_until_number(monkeypatch, capsys):
    answers = iter(["abc", "5"])
    # Das lambda braucht den Parameter prompt: input wird mit Text aufgerufen
    monkeypatch.setattr("builtins.input", lambda prompt: next(answers))
    assert ask_for_field("X") == 5
    assert "'abc' ist keine Zahl" in capsys.readouterr().out

In [ ]:
!{sys.executable} -m pytest -v test_eingabe.py

**Übung 10 (Computergegner, schwer):** Der Computer gewinnt, wenn er kann; sonst blockiert er; sonst nimmt er die Mitte; sonst ein zufälliges freies Feld. Übergib `random.Random(seed)` als Parameter, damit dein Test reproduzierbar ist.

Das lernst du: Zufall in Tests beherrschen und einen Zug an einer Kopie des Felds ausprobieren (`board.copy()` reicht nicht, siehe Notebook 06).

In [ ]:
import random

from tictactoe import EMPTY, create_board, has_won, place_mark


def free_fields(board: list[list[str]]) -> list[int]:
    return [field for field in range(1, 10) if board[(field - 1) // 3][(field - 1) % 3] == EMPTY]


def winning_move(board: list[list[str]], mark: str) -> int | None:
    for field in free_fields(board):
        trial = [row.copy() for row in board]  # Kopie: das echte Feld bleibt unverändert
        place_mark(trial, field, mark)
        if has_won(trial, mark):
            return field
    return None


def computer_move(board: list[list[str]], mark: str, other: str, rng: random.Random) -> int:
    for candidate in (winning_move(board, mark), winning_move(board, other)):
        if candidate is not None:
            return candidate
    return 5 if 5 in free_fields(board) else rng.choice(free_fields(board))


board = create_board()
for field, mark in [(1, "X"), (2, "X"), (5, "O")]:
    place_mark(board, field, mark)
print(computer_move(board, "O", "X", random.Random(1)))  # 3: blockiert X

# Reproduzierbar: Mit demselben Startwert wählt der Zufall dasselbe Feld.
board = create_board()
place_mark(board, 5, "X")
first = computer_move(board, "O", "X", random.Random(42))
assert computer_move(board, "O", "X", random.Random(42)) == first
print(first)